# Day 7 — Eager vs lazy in one picture

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** on Medium (link added after publishing)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [26]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


**Eager:** every line runs immediately and returns a `DataFrame` — like pandas.
**Lazy:** every line adds a step to a **query plan**; nothing runs until `collect()`. Before running, Polars **optimizes** the whole plan.

Question for today: *which customer states get the most late deliveries?*

In [27]:
import time

ORDERS = "data/raw/olist_orders_dataset.csv"
CUSTOMERS = "data/raw/olist_customers_dataset.csv"

## 1. Eager

In [28]:
def eager():
    orders = pl.read_csv(ORDERS, try_parse_dates=True)
    customers = pl.read_csv(CUSTOMERS)
    return (
        orders.join(customers, on="customer_id")
        .filter(pl.col("order_status") == "delivered")
        .with_columns(is_late=pl.col("order_delivered_customer_date") > pl.col("order_estimated_delivery_date"))
        .group_by("customer_state")
        .agg(orders=pl.len(), late_pct=(pl.col("is_late").mean() * 100).round(1))
        .sort("late_pct", descending=True)
    )


eager().head()

customer_state,orders,late_pct
str,u32,f64
"""AL""",397,23.9
"""MA""",717,19.7
"""PI""",476,16.0
"""CE""",1279,15.3
"""SE""",335,15.2


## 2. Lazy: same code, `scan_*` instead of `read_*`

In [29]:
def lazy():
    orders = pl.scan_csv(ORDERS, try_parse_dates=True)
    customers = pl.scan_csv(CUSTOMERS)
    return (
        orders.join(customers, on="customer_id")
        .filter(pl.col("order_status") == "delivered")
        .with_columns(is_late=pl.col("order_delivered_customer_date") > pl.col("order_estimated_delivery_date"))
        .group_by("customer_state")
        .agg(orders=pl.len(), late_pct=(pl.col("is_late").mean() * 100).round(1))
        .sort("late_pct", descending=True)
    )


plan = lazy()
print(type(plan).__name__)  # nothing has been read yet
plan.collect_schema()         # the output schema is known without running

LazyFrame


Schema([('customer_state', String), ('orders', UInt32), ('late_pct', Float64)])

## 3. What the optimizer changed

Two database words:

- **Projection** = a choice of *columns* (SQL `SELECT a, b`).
- **Predicate** = a condition on *rows* (SQL `WHERE`).

**Pushdown** moves them as close to the file reader as possible, so unneeded data is never read. Plans read **bottom up**: scans at the bottom, the final sort at the top.

Compare the plan **as written** with the plan **as executed**:

In [30]:
print(plan.explain(optimized=False))

SORT BY [descending: [true]] [col("late_pct")]
  AGGREGATE[maintain_order: false]
    [len().alias("orders"), ((col("is_late").mean() * 100.0)).round().alias("late_pct")] BY [col("customer_state")]
    FROM
     WITH_COLUMNS:
     [(col("order_delivered_customer_date") > col("order_estimated_delivery_date")).alias("is_late")] 
      FILTER col("order_status") == "delivered"
      FROM
        INNER JOIN:
        LEFT PLAN ON: [col("customer_id")]
          Csv SCAN [data/raw/olist_orders_dataset.csv]
          PROJECT */8 COLUMNS
          ESTIMATED ROWS: 99185
        RIGHT PLAN ON: [col("customer_id")]
          Csv SCAN [data/raw/olist_customers_dataset.csv]
          PROJECT */5 COLUMNS
          ESTIMATED ROWS: 102659
        END INNER JOIN


In [31]:
print(plan.explain())

SORT BY [descending: [true]] [col("late_pct")]
  AGGREGATE[maintain_order: false]
    [len().alias("orders"), ((col("is_late").mean() * 100.0)).round().alias("late_pct")] BY [col("customer_state")]
    FROM
    simple π 2/2 ["customer_state", "is_late"]
       WITH_COLUMNS:
       [(col("order_delivered_customer_date") > col("order_estimated_delivery_date")).alias("is_late")] 
        simple π 3/3 ["order_delivered_customer_date", ... 2 other columns]
          INNER JOIN:
          LEFT PLAN ON: [col("customer_id")]
            simple π 3/3 ["order_delivered_customer_date", ... 2 other columns]
              Csv SCAN [data/raw/olist_orders_dataset.csv]
              PROJECT 4/8 COLUMNS
              SELECTION: col("order_status") == "delivered"
              ESTIMATED ROWS: 99185
          RIGHT PLAN ON: [col("customer_id")]
            simple π 2/2 ["customer_state", ... 1 other column]
              Csv SCAN [data/raw/olist_customers_dataset.csv]
              PROJECT 2/5 COLUMNS
  

Things to spot in the optimized plan:

- **Projection pushdown:** `PROJECT */8` becomes `PROJECT 4/8` for orders (`customer_id`, `order_status`, two dates) and `*/5` becomes `2/5` for customers. A `simple π 3/3` right after the scan drops `order_status` once the filter has used it.
- **Predicate pushdown:** the `FILTER order_status == "delivered"` above the join moves **into the orders scan** as `SELECTION`, so fewer rows are joined.
- `ESTIMATED ROWS` is a guess from file size (99,185 vs 99,441 actual), not a count.

## 4. Timing

In [32]:
def best_of(fn, runs=5):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


results = {
    "eager": best_of(eager),
    "lazy, streaming (2.x default)": best_of(lambda: lazy().collect()),
    "lazy, in-memory": best_of(lambda: lazy().collect(engine="in-memory")),
}
for name, sec in results.items():
    print(f"{name:<32} {sec * 1000:7.1f} ms")

eager                              132.6 ms
lazy, streaming (2.x default)       56.8 ms
lazy, in-memory                    116.1 ms


## 5. Row order is not a promise

In Polars 2.x, `collect()` runs lazy queries on the **streaming engine** by default. Streaming does not guarantee row order for joins and group-bys. Without a final `sort`, two runs can return rows in different orders.

In [33]:
unsorted = (
    pl.scan_csv(CUSTOMERS)
    .group_by("customer_state")
    .agg(n=pl.len())
)
runs = [unsorted.collect()["customer_state"].to_list() for _ in range(5)]
# all() is a built-in function: True only if every run matches the first one
print("same order every run:", all(r == runs[0] for r in runs))

# The fix: say what order you want
unsorted.sort("n", descending=True).collect().head()

same order every run: False


customer_state,n
str,u32
"""SP""",41746
"""RJ""",12852
"""MG""",11635
"""RS""",5466
"""PR""",5045


## A LazyFrame is a recipe, not a dish

A LazyFrame holds **no data**, only the plan. Every method returns a **new** plan and leaves the old one unchanged, so building plans is almost free.

Every `collect()` runs the plan **from scratch**: it does not remember a previous result.

- Need a result twice? Collect once into a variable.
- Several results from one source? `pl.collect_all([...])` runs them together and can share common work.
- Reusing part of a plan inside a bigger one? That is caching, covered later in the series.


In [ ]:
base = pl.scan_csv(ORDERS)
smaller = base.head(5)  # a new plan; base is unchanged

print(base.explain())
print(smaller.explain())


## When to use which

| Use eager when | Use lazy when |
|---|---|
| exploring in a notebook | a query has several steps |
| data is small | data is large or larger than RAM |
| you need intermediate results | you only need the final result |

Rule of thumb from here on: **explore eager, ship lazy.** Any `DataFrame` becomes lazy with `.lazy()`.

## Results for the post

| Mode | Time (ms) |
|---|---|
| eager | 107.9 |
| lazy, streaming | 52.8 |
| lazy, in-memory | 129.1 |

## Try it yourself

1. Add `.head(5)` to the lazy plan before `collect()` and check `explain()`. Where does the slice end up?
2. Swap the order of `join` and `filter` in your code. Does the optimized plan change?